# Scan Geometry

In [5]:
import numpy as np


def gamma_m(w_det: float, sdd: float) -> float:
    """
    Calculate the half-angle subtended by the detector.
    """
    return np.atan(0.5 * w_det / sdd)


def critical_pitch(
    h_det: float, sdd: float, sod: float, w_det: float | None = None
) -> float:
    """
    Calculate the critical pitch for a given detector height, source-to-detector distance, and source-to-object distance.
    Optionally, the width of the detector can be provided to calculate the maximum pitch that can be resolved without aliasing.
    """
    if w_det is None:
        w_det = h_det  # assume a square detector if width is not provided

    num = np.pi * h_det * sod
    gm = gamma_m(w_det, sdd)  # half-angle subtended by the detector
    den = np.sqrt(sdd**2 + (w_det / 2.0) ** 2) * (np.pi / 2.0 + gm)

    return num / den


def excess_turn_angle(w_det: float, sdd: float) -> float:
    """
    Calculate the excess turn angle for a given pitch, detector width, and source-to-detector distance.
    """
    return np.pi + 2 * gamma_m(w_det, sdd)


def excess_turn_height(pitch: float, w_det: float, sdd: float) -> float:
    """
    Calculate the excess turn for a given pitch, detector width, and source-to-detector distance.
    """
    gm = gamma_m(w_det, sdd)  # half-angle subtended by the detector
    return pitch * excess_turn_angle(w_det, sdd) / (2 * np.pi)  # (0.5 + gm / np.pi)


def total_scan_height(h_vol: float, sdd: float, w_det: float, pitch: float) -> float:
    """
    Calculate the scan height for a given pitch, detector height, source-to-detector distance, and optionally detector width.

    This computes the total scan height needs to conver the ends of a volume of height h_fov.
    """

    excess = excess_turn_height(pitch, w_det, sdd)
    return h_vol + excess


def n_turns(h_vol: float, sdd: float, w_det: float, pitch: float) -> float:
    """
    Calculate the number of turns required to cover a volume of height h_vol with a given pitch.
    """
    return total_scan_height(h_vol, sdd, w_det, pitch) / pitch


def pitch_from_turns(n_turns: float, h_vol: float, sdd: float, w_det: float) -> float:
    """
    Calculate the pitch required to cover a volume of height h_vol with a given number of turns.
    """
    den = n_turns - excess_turn_angle(w_det, sdd) / (
        2 * np.pi
    )  # excess turn per unit pitch
    return h_vol / den
    # return (h_vol + excess * n_turns) / n_turns
    return


class ScanGeometry:
    def __init__(
        self,
        h_det: float,
        sdd: float,
        sod: float,
        pixel_size: float,
        w_det: float | None = None,
    ):
        self.h_det = h_det
        self.sdd = sdd
        self.sod = sod
        self.pixel_size = pixel_size

        # assume a square detector if width is not provided
        self.w_det = w_det if w_det is not None else h_det

    def n_turns(self, h_vol: float) -> float:
        if self.pitch is None:
            raise ValueError("Pitch must be provided to calculate number of turns.")
        h_total = h_vol + self.excess_scan_height
        return h_total / self.pitch

    @property
    def critical_pitch(self) -> float:
        # This computes the maximum pitch required for exact reconstruction per Katsevich. It is the result of the requirement that the detector captures Tam-Danielson window for each point in the volume (Noo equation 79).
        num = (np.cos(self.gamma_m) ** 2) * (self.h_det - self.pixel_size) * self.sod
        return num / (self.sdd * self.excess_scan_turn)  # critical pitch

    @property
    def excess_scan_turn(self) -> float:
        # This is the excess turn needed for reconstruction at the top and bottom of the FOV; it acccomodates both the top and bottom excess. It is a pure unitless number.
        return 0.5 + self.gamma_m / np.pi  # excess turn per unit pitch

    @property
    def excess_scan_height(self) -> float:
        # This is the excess scan height needed for reconstruction at the top and bottom of the FOV; it acccomodates both the top and bottom excess. This has units of length.
        return self.pitch * self.excess_scan_turn

    @property
    def pitch(self) -> float:
        return self.critical_pitch

    @property
    def gamma_m(self) -> float:
        # This the the 1/2 cone angle computed at the detector plane in radians
        return np.atan(0.5 * self.w_det / self.sdd)

    @property
    def gamma_m_deg(self) -> float:
        # This the the 1/2 cone angle computed at the detector plane in degrees
        return np.degrees(self.gamma_m)

    @property
    def mag_factor(self) -> float:
        # Magnification factor at the COR
        return self.sdd / self.sod


In [6]:
sdd_dict = {
    "18650": 788.0,  # source-to-detector distance
    "2170": 800.0,  # source-to-object distance
    "48xx": 775.0,  # source-to-object distance
    "coin": 788.5,
    "HR": 800,
}

sod_dict = {
    "18650": 38.0,  # source-to-object distance
    "2170": 44.0,  # source-to-object distance
    "48xx": 90.0,  # source-to-object distance
    "coin": 35.5,
    "HR": 64.0,
}

px_dict = {
    "18650": 0.3,  # source-to-object distance
    "2170": 0.3,  # source-to-object distance
    "48xx": 0.3,  # source-to-object distance
    "coin": 0.3,
    "HR": 0.15,
}

h_dict = {
    "18650": [68.644],  # source-to-object distance
    "2170": [77.092, 77.323, 82.323, 87.323, 92.323],  # source-to-object distance
    "48xx": [102.741],  # source-to-object distance
    "coin": [8.165, 1.917, 0.917, 0.417],
    "HR": [77.323, 82.323, 87.323, 92.323],
}


In [12]:
h_det = 427.5
scan_object = "coin"
scan_geom = ScanGeometry(
    h_det=h_det,  # height of the detector
    sdd=sdd_dict[scan_object],  # source-to-detector distance
    sod=sod_dict[scan_object],  # source-to-object distance
    pixel_size=px_dict[scan_object],  # pixel size
)
heights = h_dict[scan_object]
print(f"Critical pitch: {scan_geom.critical_pitch:.3f}")
print(f"Half cone angle (deg): {scan_geom.gamma_m_deg:.3f}")
print(f"Half cone angle (rad): {scan_geom.gamma_m:.3f}")
print(f"Pitch: {scan_geom.pitch:.3f}")
for h_vol in heights:
    print(f"Predicted turns for height {h_vol}: {scan_geom.n_turns(h_vol):.3f}")


Critical pitch: 30.666
Half cone angle (deg): 15.167
Half cone angle (rad): 0.265
Pitch: 30.666
Predicted turns for height 8.165: 0.851
Predicted turns for height 1.917: 0.647
Predicted turns for height 0.917: 0.614
Predicted turns for height 0.417: 0.598


In [13]:
def adjust(x):
  return 1.0159*x + 0.0668

In [18]:
vals = [2.79,2.75,1.95,0.94,2.76,2.89,3.03,3.16,2.08,2.18,2.27,2.36,0.73,0.7,0.68]
pred_vals = [2.675,2.633,1.891,0.851,2.639,2.772,2.905,3.038,1.996,2.087,2.179,2.270,0.647,0.614,0.598]

for val, pred_val in zip(vals, pred_vals):
    print(f"Adjusted value: {adjust(pred_val)}")
    print(f"Relative error: {(adjust(pred_val)-val)/val:.3%}")
    print()

Adjusted value: 2.7843325
Relative error: -0.203%

Adjusted value: 2.7416647000000003
Relative error: -0.303%

Adjusted value: 1.9878669
Relative error: 1.942%

Adjusted value: 0.9313309
Relative error: -0.922%

Adjusted value: 2.7477601000000003
Relative error: -0.443%

Adjusted value: 2.8828748
Relative error: -0.247%

Adjusted value: 3.0179895
Relative error: -0.396%

Adjusted value: 3.1531042
Relative error: -0.218%

Adjusted value: 2.0945364000000004
Relative error: 0.699%

Adjusted value: 2.1869833000000005
Relative error: 0.320%

Adjusted value: 2.2804461000000003
Relative error: 0.460%

Adjusted value: 2.3728930000000004
Relative error: 0.546%

Adjusted value: 0.7240873
Relative error: -0.810%

Adjusted value: 0.6905626
Relative error: -1.348%

Adjusted value: 0.6743081999999999
Relative error: -0.837%

